# Extracción de Datos

Extrae datos de forma confiable desde archivos, bases de datos, APIs y páginas web

## Introducción

La extracción es la primera fase de cualquier pipeline ETL: obtener los datos desde sus fuentes originales. Las fuentes pueden ser archivos planos (CSV, JSON, Excel), bases de datos (SQLite, PostgreSQL), APIs web (REST, GraphQL) o páginas web (web scraping). Cada fuente tiene sus particularidades: encodings, paginación, límites de rate, estructuras anidadas. Dominar la extracción robusta — con manejo de errores, encodings y grandes volúmenes — es esencial para cualquier Data Engineer.

### Objetivos de Aprendizaje

- Leer CSV, JSON y Excel con pandas de forma robusta
- Conectar a SQLite con sqlite3 y pandas para extraer datos tabulares
- Hacer peticiones HTTP a APIs REST y parsear la respuesta JSON
- Realizar scraping básico con requests para obtener datos de páginas web
- Manejar problemas de encoding y leer archivos grandes en chunks

## Leyendo CSV, JSON y Excel con pandas

> pandas ofrece read_csv(), read_json() y read_excel() como las funciones principales de lectura. Cada una tiene parámetros clave: sep para el separador en CSV, orient para la estructura en JSON, y sheet_name para Excel multi-hoja. En casos reales, los archivos llegan con problemas: encodings mixtos, separadores no estándar, filas de cabecera adicionales, o celdas fusionadas en Excel.

In [ ]:
import pandas as pd

# CSV básico con separador coma y encoding UTF-8
df_csv = pd.read_csv('ventas.csv')

# CSV con opciones avanzadas
df_csv_avanzado = pd.read_csv(
    'ventas_europa.csv',
    sep=';',
    encoding='latin-1',
    decimal=',',
    thousands='.',
    skiprows=2,
    na_values=['N/A', '-', ''],
    parse_dates=['fecha_venta']
)

# JSON como lista de objetos (el más común en APIs)
df_json = pd.read_json('usuarios.json', orient='records')

# JSON anidado: necesita normalización
import json
with open('pedidos.json', 'r', encoding='utf-8') as f:
    datos = json.load(f)
df_pedidos = pd.json_normalize(datos,
                                record_path='items',
                                meta=['pedido_id', 'fecha'])

# Excel: una sola hoja
df_excel = pd.read_excel('reporte.xlsx', sheet_name='Ventas')

# Todas las hojas como diccionario
hojas = pd.read_excel('reporte.xlsx', sheet_name=None)
for nombre_hoja, df_hoja in hojas.items():
    print(f"Hoja '{nombre_hoja}': {len(df_hoja)} filas")

# Excel con opciones: saltar filas, seleccionar columnas
df_excel_limpio = pd.read_excel(
    'reporte.xlsx',
    sheet_name='Q1_2024',
    header=2,
    usecols='A:F',
    nrows=1000
)

## Extracción desde SQLite con sqlite3 y pandas

> SQLite es la base de datos más usada en pipelines ETL locales y en testing: no requiere servidor, es un solo archivo. La combinación de sqlite3 (biblioteca estándar de Python) con pd.read_sql() o pd.read_sql_query() permite extraer datos directamente a DataFrames. También puedes usar parámetros parametrizados para evitar SQL injection cuando los valores vienen de variables.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.execute("""CREATE TABLE productos (
    id INTEGER PRIMARY KEY,
    nombre TEXT,
    precio REAL
)""")
conn.execute("""CREATE TABLE ventas (
    id INTEGER PRIMARY KEY,
    producto_id INTEGER,
    cantidad INTEGER,
    precio REAL,
    region TEXT
)""")
conn.executemany("INSERT INTO productos VALUES (?,?,?)", [
    (1, 'Laptop', 1200), (2, 'Mouse', 25), (3, 'Monitor', 400)
])
conn.executemany("INSERT INTO ventas VALUES (?,?,?,?,?)", [
    (1, 1, 2, 1200, 'Norte'), (2, 2, 5, 25, 'Sur'),
    (3, 1, 1, 1200, 'Norte'), (4, 3, 2, 400, 'Sur')
])
conn.commit()

df_productos = pd.read_sql('SELECT * FROM productos', conn)
print(f"Productos: {len(df_productos)} filas")

df_ventas_norte = pd.read_sql("""
    SELECT v.id, p.nombre, v.cantidad, v.precio,
           v.cantidad * v.precio AS total, v.region
    FROM ventas v
    JOIN productos p ON v.producto_id = p.id
    WHERE v.region = 'Norte'
    ORDER BY v.id
""", conn)
print(f"Ventas región Norte: {len(df_ventas_norte)} filas")
print(df_ventas_norte)

region_objetivo = 'Sur'
df_parametrizado = pd.read_sql(
    "SELECT * FROM ventas WHERE region = ?",
    conn,
    params=(region_objetivo,)
)
print(f"\nVentas región Sur: {len(df_parametrizado)} filas")

conn.close()

## Extracción desde APIs REST con requests

> Las APIs REST son una de las fuentes de datos más comunes. requests.get() realiza peticiones HTTP y .json() parsea la respuesta. En la práctica, hay que manejar: paginación (offset/cursor), rate limiting (429), errores de red (timeout), y autenticación (API keys, Bearer tokens). La función de extracción debe ser resiliente a estos problemas.

In [ ]:
import requests
import pandas as pd
import time

# Petición básica a una API REST
url = 'https://jsonplaceholder.typicode.com/posts'
respuesta = requests.get(url, timeout=10)
respuesta.raise_for_status()

datos = respuesta.json()
df_posts = pd.DataFrame(datos)
print(f"Posts obtenidos: {len(df_posts)}")
print(df_posts[['id', 'userId', 'title']].head(3))

# API con autenticación y parámetros
API_KEY = "mi_api_key_secreta"
params = {
    'start_date': '2024-01-01',
    'end_date':   '2024-03-31',
    'limit':      100,
    'offset':     0
}
headers = {
    'Authorization': f'Bearer {API_KEY}',
    'Content-Type': 'application/json'
}

# Extracción con paginación automática
def extraer_api_paginada(url_base: str, tam_pagina: int = 100) -> pd.DataFrame:
    todos_los_datos = []
    offset = 0

    while True:
        respuesta = requests.get(
            url_base,
            params={'limit': tam_pagina, 'offset': offset},
            timeout=10
        )
        respuesta.raise_for_status()
        payload = respuesta.json()
        pagina = payload.get('data', payload)

        if not pagina:
            break

        todos_los_datos.extend(pagina)
        print(f"  Página offset={offset}: {len(pagina)} registros")

        if len(pagina) < tam_pagina:
            break

        offset += tam_pagina
        time.sleep(0.1)

    print(f"Total extraído: {len(todos_los_datos)} registros")
    return pd.DataFrame(todos_los_datos)

## Scraping Básico con requests + BeautifulSoup

> Cuando no hay API disponible, el web scraping permite extraer datos de páginas HTML. requests descarga el HTML y BeautifulSoup lo parsea como árbol DOM. Los selectores CSS (select()) y los métodos find() / find_all() permiten navegar la estructura. Importante: siempre revisar el archivo robots.txt del sitio, respetar el rate limit, y verificar los términos de servicio.

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Scraping básico de tabla HTML
url = 'https://es.wikipedia.org/wiki/Anexo:Pa%C3%ADses_por_superficie'
headers = {
    'User-Agent': 'Mozilla/5.0 (compatible; DataPipeline/1.0)'
}
respuesta = requests.get(url, headers=headers, timeout=15)
respuesta.raise_for_status()

soup = BeautifulSoup(respuesta.text, 'html.parser')
tabla = soup.find('table', class_='wikitable')

cabeceras = [th.get_text(strip=True) for th in tabla.find_all('th')]

filas = []
for tr in tabla.find_all('tr')[1:]:
    celdas = [td.get_text(strip=True) for td in tr.find_all('td')]
    if celdas:
        filas.append(celdas)

df_paises = pd.DataFrame(filas, columns=cabeceras[:len(filas[0])])
print(df_paises.head(5))

# pandas puede parsear tablas HTML directamente
tablas = pd.read_html(url)
df_tabla_0 = tablas[0]
print(f"\nTablas encontradas: {len(tablas)}")

# Scraping de elementos específicos (no tablas)
html_ejemplo = """
<div class=\"producto\">
    <h2 class=\"nombre\">Laptop ProMax</h2>
    <span class=\"precio\">1299.99</span>
    <span class=\"stock\">En stock</span>
</div>
<div class=\"producto\">
    <h2 class=\"nombre\">Mouse Ergonómico</h2>
    <span class=\"precio\">45.50</span>
    <span class=\"stock\">Sin stock</span>
</div>
"""
soup2 = BeautifulSoup(html_ejemplo, 'html.parser')
productos = []
for div in soup2.find_all('div', class_='producto'):
    productos.append({
        'nombre': div.find('h2', class_='nombre').get_text(strip=True),
        'precio': float(div.find('span', class_='precio').get_text(strip=True)),
        'stock':  div.find('span', class_='stock').get_text(strip=True)
    })
df_productos = pd.DataFrame(productos)
print(df_productos)

## Encoding y Lectura de Archivos Grandes en Chunks

> Dos problemas frecuentes en extracción real: encoding incorrecto (el archivo dice ser UTF-8 pero usa latin-1) y archivos tan grandes que no caben en memoria. Para el encoding, chardet detecta automáticamente el correcto. Para archivos grandes, pd.read_csv() con chunksize retorna un iterador de DataFrames parciales que se procesan uno a uno, manteniendo bajo el uso de memoria.

In [ ]:
import pandas as pd
import chardet

# Detectar encoding automáticamente
def detectar_encoding(ruta: str, muestra_bytes: int = 50000) -> str:
    with open(ruta, 'rb') as f:
        muestra = f.read(muestra_bytes)
    resultado = chardet.detect(muestra)
    encoding_detectado = resultado['encoding']
    confianza = resultado['confidence']
    print(f"Encoding detectado: {encoding_detectado} (confianza: {confianza:.0%})")
    return encoding_detectado

def leer_csv_robusto(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, encoding='utf-8')
    except UnicodeDecodeError:
        print("UTF-8 falló, detectando encoding...")
        enc = detectar_encoding(ruta)
        return pd.read_csv(ruta, encoding=enc)

# Leer archivos grandes en chunks
def procesar_csv_grande(ruta: str, tam_chunk: int = 100_000) -> pd.DataFrame:
    chunks_procesados = []
    total_filas = 0

    for i, chunk in enumerate(pd.read_csv(ruta, chunksize=tam_chunk)):
        chunk_limpio = chunk.dropna(subset=['id', 'monto'])
        chunk_limpio = chunk_limpio[chunk_limpio['monto'] > 0]
        chunks_procesados.append(chunk_limpio)
        total_filas += len(chunk_limpio)
        print(f"Chunk {i+1}: {len(chunk)} filas leídas → {len(chunk_limpio)} válidas")

    resultado = pd.concat(chunks_procesados, ignore_index=True)
    print(f"\nTotal procesado: {total_filas} filas válidas")
    return resultado

# Agregar directamente por chunks (sin guardar todo en RAM)
def agregar_por_chunks(ruta: str, tam_chunk: int = 50_000) -> pd.DataFrame:
    totales = {}

    for chunk in pd.read_csv(ruta, chunksize=tam_chunk):
        chunk['monto'] = pd.to_numeric(chunk['monto'], errors='coerce').fillna(0)
        for cat, grupo in chunk.groupby('categoria'):
            if cat not in totales:
                totales[cat] = {'suma': 0, 'conteo': 0}
            totales[cat]['suma']   += grupo['monto'].sum()
            totales[cat]['conteo'] += len(grupo)

    df_totales = pd.DataFrame.from_dict(totales, orient='index').reset_index()
    df_totales.columns = ['categoria', 'total_ventas', 'num_transacciones']
    return df_totales.sort_values('total_ventas', ascending=False)

## Pipeline de Extracción Multi-Fuente

Extrae datos de tres fuentes distintas (CSV, SQLite y API simulada) y los combina en un único DataFrame.

In [ ]:
import pandas as pd
import sqlite3
import json
import io

# Fuente 1: CSV
csv_productos = """id_producto,nombre,precio,categoria
P001,Laptop Pro,1200.00,Electrónica
P002,Mouse Inalámbrico,25.50,Periféricos
P003,Teclado Mecánico,85.00,Periféricos
P004,Monitor 27\"",350.00,Electrónica
"""
df_productos = pd.read_csv(io.StringIO(csv_productos))
print(f"✓ CSV: {len(df_productos)} productos")

# Fuente 2: SQLite
conn = sqlite3.connect(':memory:')
conn.execute("""CREATE TABLE ventas (
    id INT, producto_id TEXT, cantidad INT, fecha TEXT, vendedor TEXT
)""")
conn.executemany("INSERT INTO ventas VALUES (?,?,?,?,?)", [
    (1, 'P001', 2, '2024-01-15', 'María'),
    (2, 'P002', 5, '2024-01-15', 'Carlos'),
    (3, 'P003', 3, '2024-01-16', 'María'),
    (4, 'P001', 1, '2024-01-17', 'Ana'),
    (5, 'P004', 2, '2024-01-18', 'Carlos'),
])
conn.commit()
df_ventas = pd.read_sql("SELECT id, producto_id, cantidad, fecha, vendedor FROM ventas ORDER BY fecha", conn)
conn.close()
print(f"✓ SQLite: {len(df_ventas)} ventas")

# Fuente 3: JSON (simulando respuesta de API)
api_response = json.dumps([
    {"id_vendedor": "María",  "region": "Norte", "meta_mensual": 5000},
    {"id_vendedor": "Carlos", "region": "Sur",   "meta_mensual": 4500},
    {"id_vendedor": "Ana",    "region": "Este",  "meta_mensual": 4000},
])
df_vendedores = pd.read_json(io.StringIO(api_response), orient='records')
print(f"✓ JSON/API: {len(df_vendedores)} vendedores")

# Combinar las tres fuentes
df_combinado = (
    df_ventas
    .merge(df_productos, left_on='producto_id', right_on='id_producto')
    .merge(df_vendedores, left_on='vendedor', right_on='id_vendedor')
)
df_combinado['total'] = df_combinado['cantidad'] * df_combinado['precio']

print("\n=== DATOS COMBINADOS ===")
cols = ['fecha', 'nombre', 'cantidad', 'precio', 'total', 'vendedor', 'region']
print(df_combinado[cols].to_string(index=False))
print(f"\nTotal de ventas: ${df_combinado['total'].sum():,.2f}")

## Tips y Mejores Prácticas

> Siempre usa timeout= en requests.get(). Sin él, si el servidor no responde, tu script se cuelga indefinidamente. Para APIs normales, timeout=10-30 segundos es razonable; para descargas grandes, usa timeout=(5, 60) — (connect_timeout, read_timeout).

> pd.read_csv() por defecto infiere los tipos de columna. Esto puede causar sorpresas: IDs con ceros a la izquierda se convierten a int perdiéndolos (0001 → 1). Usa dtype={"id": str} para columnas que deben mantenerse como strings.

> Para archivos CSV mayores de 500MB, usa chunksize=100_000 en read_csv(). Procesa y agrega cada chunk antes de pasar al siguiente. Esto te permite trabajar con archivos de varios GB con menos de 1GB de RAM.

> Cuando hagas scraping, agrega un time.sleep(1) entre peticiones para no sobrecargar el servidor. Algunos sitios detectan y bloquean bots por hacer demasiadas peticiones por segundo. También incluye un User-Agent en los headers para identificar tu bot.

## Errores Comunes

### No manejar errores HTTP en peticiones a APIs

¿Por qué ocurre?
- requests.get() no lanza excepción ante errores HTTP (404, 500, 429). El script continúa con datos incorrectos o vacíos sin que el programador se dé cuenta.

Solución
- Siempre llama a respuesta.raise_for_status() inmediatamente después de requests.get(). Esto lanza HTTPError automáticamente ante cualquier código de estado 4xx o 5xx.

### Leer archivos grandes directamente sin chunking

¿Por qué ocurre?
- df = pd.read_csv("archivo_5GB.csv") intenta cargar todo en RAM. Con 8GB de RAM y un archivo de 5GB, el proceso falla con MemoryError o hace swap a disco, volviendo todo lentísimo.

Solución
- Usa pd.read_csv(..., chunksize=100_000) para archivos > 500MB. Si solo necesitas un resumen, puedes agregar directamente dentro del loop de chunks sin acumular DataFrames.

### Asumir que el encoding de un CSV es siempre UTF-8

¿Por qué ocurre?
- Archivos exportados desde Excel, sistemas SAP o bases de datos Windows frecuentemente usan Windows-1252 o latin-1. read_csv() con encoding="utf-8" falla con UnicodeDecodeError en el primer carácter especial.

Solución
- Intenta UTF-8 primero; atrapa UnicodeDecodeError y reintenta con latin-1. Para mayor robustez, usa chardet para detectar automáticamente el encoding.

### Hacer scraping sin respetar robots.txt ni rate limits

¿Por qué ocurre?
- Hacer 1000 peticiones por segundo a un servidor puede causar un ban de IP, violar los términos de servicio, o en casos extremos, acciones legales. Muchos sitios bloquean IPs que hacen demasiadas peticiones.

Solución
- Revisa siempre https://sitio.com/robots.txt antes de scrapear. Agrega time.sleep(1) entre peticiones. Identifica tu bot con un User-Agent descriptivo. Usa la API oficial si existe.